In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# =====================================================================
# Speaker-level GroupKFold cross-validation for RAVDESS and RESD.
#
# IMPORTANT:
# - The OUTER GroupKFold defines held-out test speakers.
# - A separate INNER GroupKFold split within the outer-training speakers
#   is used only for early stopping / checkpoint selection.
# - Final metrics are always computed on the OUTER held-out test speakers.
# - The final reported SD is computed across the 5 OUTER fold means
#   (after averaging random seeds within each fold).
#
# Scope:
# This supplementary sensitivity analysis covers the four utterance-level
# models (Handcrafted SVM-RBF, Handcrafted MLP, emotion2vec MLP, and
# Concat Fusion MLP). Sequence-level Cross-Attention is not included here.
# =====================================================================

import os
import json
import random
import pickle
import warnings
import numpy as np
import pandas as pd

from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, recall_score

SEEDS = [42, 123, 2024]
N_SPLITS = 5

BASE_PROJECT = Path("/content/drive/MyDrive/New Jurnal Cross")

HC_DIR = BASE_PROJECT / "processed_intra_features_hc_noaug"
E2V_DIR = BASE_PROJECT / "processed_intra_features_e2v_plus_base"

OUT_DIR = BASE_PROJECT / "results_speaker_level_cv"
OUT_DIR.mkdir(parents=True, exist_ok=True)

LABELS = ["angry", "disgust", "fear", "happy", "neutral", "sad"]
NUM_CLASSES = len(LABELS)

DATASETS = ["ravdess", "resd"]  # EmoDB 2.0 excluded: official speaker split

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


Device: cuda


In [3]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def load_pooled_corpus(feature_dir, dataset_name, x_prefix):
    """
    Re-pool the existing cached train/val/test arrays + meta CSVs into the
    full corpus, then repartition by speaker. No corpus-level scaling
    statistics are reused; scaling is fitted separately inside each fold.
    """
    ds_dir = feature_dir / dataset_name

    Xs, ys, speakers, metas = [], [], [], []
    for split in ["train", "val", "test"]:
        X = np.load(ds_dir / f"X_{x_prefix}_{split}.npy").astype(np.float32)
        y = np.load(ds_dir / f"y_{split}.npy").astype(np.int64)
        meta = pd.read_csv(ds_dir / f"meta_{split}.csv")

        assert len(X) == len(y) == len(meta), (
            f"Length mismatch in {dataset_name} {split}: "
            f"X={len(X)} y={len(y)} meta={len(meta)}"
        )

        Xs.append(X)
        ys.append(y)
        speakers.append(meta["speaker"].astype(str).values)
        metas.append(meta)

    X_all = np.concatenate(Xs, axis=0)
    y_all = np.concatenate(ys, axis=0)
    speaker_all = np.concatenate(speakers, axis=0)
    meta_all = pd.concat(metas, ignore_index=True)

    print(
        f"{dataset_name} ({x_prefix}): pooled N={len(X_all)}, "
        f"speakers={len(set(speaker_all))}, dim={X_all.shape[1]}"
    )
    return X_all, y_all, speaker_all, meta_all


def class_weights_from_training(y_train, num_classes=NUM_CLASSES):
    """Balanced class weights computed ONLY from the current training subset."""
    counts = np.bincount(y_train, minlength=num_classes).astype(np.float64)
    if np.any(counts == 0):
        raise ValueError(f"At least one class is absent from training subset: counts={counts}")
    weights = len(y_train) / (num_classes * counts)
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)


In [4]:
# ---- Model definitions (identical to notebooks 04/06/07) ----

class HandcraftedMLP(nn.Module):
    def __init__(self, input_dim=548, hidden_dim=256, num_classes=6, dropout=0.30):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.BatchNorm1d(hidden_dim // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes)
        )
    def forward(self, x): return self.net(x)


class Emotion2VecMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, num_classes=6, dropout=0.30):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.LayerNorm(hidden_dim // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes)
        )
    def forward(self, x): return self.net(x)


class ConcatFusionMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=512, num_classes=6, dropout=0.35):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.LayerNorm(hidden_dim // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, hidden_dim // 4), nn.LayerNorm(hidden_dim // 4), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim // 4, num_classes)
        )
    def forward(self, x): return self.net(x)


def make_loader(X, y, batch_size, shuffle):
    ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


def run_one_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    all_preds, all_targets = [], []

    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        if is_train:
            optimizer.zero_grad()
        with torch.set_grad_enabled(is_train):
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            if is_train:
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * X_batch.size(0)
        all_preds.append(logits.argmax(dim=1).detach().cpu().numpy())
        all_targets.append(y_batch.detach().cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)
    avg_loss = total_loss / len(loader.dataset)
    macro_f1 = f1_score(all_targets, all_preds, average="macro", zero_division=0)
    return avg_loss, macro_f1, all_preds, all_targets


def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "uar": recall_score(y_true, y_pred, average="macro", zero_division=0),
    }

In [5]:
def train_eval_mlp_fold(
    model_class, model_kwargs,
    X_train, y_train,
    X_val, y_val,
    X_test, y_test,
    seed,
    batch_size=32,
    lr=1e-3,
    weight_decay=1e-4,
    max_epochs=150,
    patience=20,
):
    """
    Fit on INNER training speakers, select the checkpoint using INNER
    validation speakers, then report metrics ONLY on OUTER held-out test
    speakers.
    """
    set_seed(seed)

    # Source/training-only scaling.
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train).astype(np.float32)
    X_val_s = scaler.transform(X_val).astype(np.float32)
    X_test_s = scaler.transform(X_test).astype(np.float32)

    model = model_class(input_dim=X_train.shape[1], **model_kwargs).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss(weight=class_weights_from_training(y_train))

    # Macro-F1 is maximized, so ReduceLROnPlateau must use mode="max".
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=5
    )

    train_loader = make_loader(X_train_s, y_train, batch_size, shuffle=True)
    val_loader = make_loader(X_val_s, y_val, batch_size, shuffle=False)
    test_loader = make_loader(X_test_s, y_test, batch_size, shuffle=False)

    best_val_f1 = -1.0
    best_state = None
    best_epoch = None
    epochs_no_improve = 0

    for epoch in range(max_epochs):
        run_one_epoch(model, train_loader, criterion, optimizer)
        _, val_f1, _, _ = run_one_epoch(model, val_loader, criterion)
        scheduler.step(val_f1)

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_epoch = epoch + 1
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                break

    if best_state is None:
        raise RuntimeError("No checkpoint was selected.")

    model.load_state_dict(best_state)
    model.to(DEVICE)

    # CRITICAL: evaluate on OUTER held-out speakers.
    _, _, test_preds, test_true = run_one_epoch(model, test_loader, criterion)
    metrics = compute_metrics(test_true, test_preds)
    metrics["best_val_macro_f1"] = best_val_f1
    metrics["selected_epoch"] = best_epoch
    return metrics


def train_eval_svm_fold(X_train, y_train, X_test, y_test):
    """
    Deterministic SVM evaluation on the OUTER held-out speakers.
    Scaling is fitted only on the outer-training data.
    """
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    model = SVC(
        kernel="rbf",
        C=10.0,
        gamma="scale",
        class_weight="balanced",
    )
    model.fit(X_train_s, y_train)
    preds = model.predict(X_test_s)
    return compute_metrics(y_test, preds)


In [6]:
# ---- Main speaker-level nested CV loop ----

all_rows = []

for dataset_name in DATASETS:
    print("=" * 100)
    print(f"SPEAKER-LEVEL CV: {dataset_name.upper()}")
    print("=" * 100)

    X_hc, y_hc, speaker_hc, meta_hc = load_pooled_corpus(HC_DIR, dataset_name, "hc")
    X_e2v, y_e2v, speaker_e2v, meta_e2v = load_pooled_corpus(E2V_DIR, dataset_name, "e2v")

    # Alignment checks.
    assert len(X_hc) == len(X_e2v), "HC and emotion2vec pools differ in length."
    assert np.array_equal(y_hc, y_e2v), "HC and emotion2vec labels are not aligned."
    assert np.array_equal(speaker_hc, speaker_e2v), "HC and emotion2vec speaker order is not aligned."

    X_concat = np.concatenate([X_e2v, X_hc], axis=1)

    n_speakers = len(set(speaker_hc))
    n_splits_this = min(N_SPLITS, n_speakers)
    if n_splits_this < N_SPLITS:
        print(f"WARNING: only {n_speakers} speakers; using {n_splits_this} folds.")

    outer_gkf = GroupKFold(n_splits=n_splits_this)

    for fold_idx, (outer_train_idx, outer_test_idx) in enumerate(
        outer_gkf.split(X_hc, y_hc, groups=speaker_hc)
    ):
        outer_train_speakers = set(speaker_hc[outer_train_idx])
        outer_test_speakers = set(speaker_hc[outer_test_idx])
        assert outer_train_speakers.isdisjoint(outer_test_speakers)

        # Inner speaker-level split ONLY for early stopping.
        inner_groups = speaker_hc[outer_train_idx]
        n_inner_speakers = len(set(inner_groups))
        inner_gkf = GroupKFold(n_splits=min(5, n_inner_speakers))

        inner_train_rel, inner_val_rel = next(
            inner_gkf.split(
                X_hc[outer_train_idx],
                y_hc[outer_train_idx],
                groups=inner_groups,
            )
        )
        inner_train_idx = outer_train_idx[inner_train_rel]
        inner_val_idx = outer_train_idx[inner_val_rel]

        assert set(speaker_hc[inner_train_idx]).isdisjoint(set(speaker_hc[inner_val_idx]))
        assert set(speaker_hc[inner_val_idx]).isdisjoint(outer_test_speakers)

        print(
            f"\nFold {fold_idx+1}/{n_splits_this} | "
            f"outer test speakers={sorted(outer_test_speakers)} | "
            f"inner train N={len(inner_train_idx)} | "
            f"inner val N={len(inner_val_idx)} | "
            f"outer test N={len(outer_test_idx)}"
        )

        # SVM is deterministic: run once per outer fold.
        svm_metrics = train_eval_svm_fold(
            X_hc[outer_train_idx], y_hc[outer_train_idx],
            X_hc[outer_test_idx], y_hc[outer_test_idx],
        )
        all_rows.append({
            "dataset": dataset_name,
            "model": "Handcrafted SVM-RBF",
            "fold": fold_idx + 1,
            "seed": "deterministic",
            **svm_metrics,
        })

        for seed in SEEDS:
            # Handcrafted MLP
            m = train_eval_mlp_fold(
                HandcraftedMLP, {},
                X_hc[inner_train_idx], y_hc[inner_train_idx],
                X_hc[inner_val_idx], y_hc[inner_val_idx],
                X_hc[outer_test_idx], y_hc[outer_test_idx],
                seed,
                batch_size=32, lr=1e-3, max_epochs=150, patience=20,
            )
            all_rows.append({
                "dataset": dataset_name, "model": "Handcrafted MLP",
                "fold": fold_idx + 1, "seed": seed, **m
            })

            # emotion2vec MLP
            m = train_eval_mlp_fold(
                Emotion2VecMLP, {},
                X_e2v[inner_train_idx], y_e2v[inner_train_idx],
                X_e2v[inner_val_idx], y_e2v[inner_val_idx],
                X_e2v[outer_test_idx], y_e2v[outer_test_idx],
                seed,
                batch_size=32, lr=1e-3, max_epochs=150, patience=20,
            )
            all_rows.append({
                "dataset": dataset_name, "model": "emotion2vec MLP",
                "fold": fold_idx + 1, "seed": seed, **m
            })

            # Concat Fusion MLP
            m = train_eval_mlp_fold(
                ConcatFusionMLP, {},
                X_concat[inner_train_idx], y_hc[inner_train_idx],
                X_concat[inner_val_idx], y_hc[inner_val_idx],
                X_concat[outer_test_idx], y_hc[outer_test_idx],
                seed,
                batch_size=32, lr=8e-4, max_epochs=150, patience=20,
            )
            all_rows.append({
                "dataset": dataset_name, "model": "Concat Fusion MLP",
                "fold": fold_idx + 1, "seed": seed, **m
            })

cv_results = pd.DataFrame(all_rows)
cv_results.to_csv(OUT_DIR / "speaker_level_cv_all_results_FIXED.csv", index=False)
display(cv_results)


SPEAKER-LEVEL CV: RAVDESS
ravdess (hc): pooled N=1056, speakers=24, dim=548
ravdess (e2v): pooled N=1056, speakers=24, dim=768

Fold 1/5 | outer test speakers=['12', '17', '21', '4', '9'] | inner train N=660 | inner val N=176 | outer test N=220

Fold 2/5 | outer test speakers=['11', '16', '20', '3', '8'] | inner train N=660 | inner val N=176 | outer test N=220

Fold 3/5 | outer test speakers=['10', '15', '2', '24', '7'] | inner train N=660 | inner val N=176 | outer test N=220

Fold 4/5 | outer test speakers=['1', '14', '19', '23', '6'] | inner train N=660 | inner val N=176 | outer test N=220

Fold 5/5 | outer test speakers=['13', '18', '22', '5'] | inner train N=704 | inner val N=176 | outer test N=176
SPEAKER-LEVEL CV: RESD
resd (hc): pooled N=1198, speakers=50, dim=548
resd (e2v): pooled N=1198, speakers=50, dim=768

Fold 1/5 | outer test speakers=['10', '22', '25', '3', '32', '35', '40', '45', '47', '9'] | inner train N=763 | inner val N=193 | outer test N=242

Fold 2/5 | outer test

,dataset,model,fold,seed,accuracy,macro_f1,weighted_f1,uar,best_val_macro_f1,selected_epoch
0,ravdess,Handcrafted SVM-RBF,1,deterministic,0.486364,0.494389,0.490848,0.479167,NaN,NaN
1,ravdess,Handcrafted MLP,1,42,0.463636,0.466278,0.462245,0.475000,0.584877,5.0
2,ravdess,emotion2vec MLP,1,42,0.881818,0.874526,0.884098,0.891667,0.924747,3.0
3,ravdess,Concat Fusion MLP,1,42,0.881818,0.876221,0.880779,0.887500,0.933325,7.0
4,ravdess,Handcrafted MLP,1,123,0.427273,0.426732,0.424617,0.429167,0.570904,11.0
...,...,...,...,...,...,...,...,...,...,...
95,resd,emotion2vec MLP,5,123,0.768908,0.726416,0.779763,0.775641,0.557297,3.0
96,resd,Concat Fusion MLP,5,123,0.752101,0.723711,0.764906,0.761316,0.541064,1.0
97,resd,Handcrafted MLP,5,2024,0.155462,0.140675,0.139396,0.216883,0.178736,6.0
98,resd,emotion2vec MLP,5,2024,0.798319,0.752811,0.806161,0.777296,0.568828,1.0


In [7]:
# ---- Correct summary: variability ACROSS OUTER SPEAKER FOLDS ----
#
# First average seeds within each outer fold. Then compute mean ± SD
# across the 5 outer-fold means. This keeps speaker-partition variability
# conceptually separate from random-seed variability.

fold_summary = (
    cv_results
    .groupby(["dataset", "model", "fold"], as_index=False)
    .agg(
        macro_f1=("macro_f1", "mean"),
        accuracy=("accuracy", "mean"),
        uar=("uar", "mean"),
        weighted_f1=("weighted_f1", "mean"),
        n_seed_runs=("seed", "count"),
    )
)

summary = (
    fold_summary
    .groupby(["dataset", "model"], as_index=False)
    .agg(
        macro_f1_mean=("macro_f1", "mean"),
        macro_f1_fold_std=("macro_f1", "std"),
        accuracy_mean=("accuracy", "mean"),
        accuracy_fold_std=("accuracy", "std"),
        uar_mean=("uar", "mean"),
        uar_fold_std=("uar", "std"),
        weighted_f1_mean=("weighted_f1", "mean"),
        weighted_f1_fold_std=("weighted_f1", "std"),
        n_outer_folds=("fold", "count"),
    )
)

# Optional seed-instability diagnostic for neural models.
neural = cv_results[cv_results["seed"].astype(str) != "deterministic"].copy()
seed_within_fold = (
    neural
    .groupby(["dataset", "model", "fold"], as_index=False)
    .agg(seed_sd_macro_f1=("macro_f1", "std"))
)
seed_variability = (
    seed_within_fold
    .groupby(["dataset", "model"], as_index=False)
    .agg(mean_within_fold_seed_sd_macro_f1=("seed_sd_macro_f1", "mean"))
)

summary = summary.merge(seed_variability, on=["dataset", "model"], how="left")

fold_summary.to_csv(OUT_DIR / "speaker_level_cv_fold_summary_FIXED.csv", index=False)
summary.to_csv(OUT_DIR / "speaker_level_cv_summary_FIXED.csv", index=False)

display(fold_summary)
display(summary)

print("\nReport macro_f1_mean ± macro_f1_fold_std as the speaker-partition sensitivity result.")
print("Do NOT describe the SD across all fold×seed runs as an SD 'across folds'.")
print("This notebook evaluates four utterance-level models; Cross-Attention is outside this supplementary CV.")


,dataset,model,fold,macro_f1,accuracy,uar,weighted_f1,n_seed_runs
0,ravdess,Concat Fusion MLP,1,0.873482,0.878788,0.886111,0.878614,3
1,ravdess,Concat Fusion MLP,2,0.945970,0.943939,0.947222,0.944051,3
2,ravdess,Concat Fusion MLP,3,0.940907,0.946970,0.950000,0.948071,3
3,ravdess,Concat Fusion MLP,4,0.994444,0.993939,0.994444,0.993939,3
4,ravdess,Concat Fusion MLP,5,0.962195,0.967803,0.960069,0.967556,3
5,ravdess,Handcrafted MLP,1,0.443186,0.440909,0.445833,0.440011,3
6,ravdess,Handcrafted MLP,2,0.555471,0.571212,0.563889,0.563388,3
7,ravdess,Handcrafted MLP,3,0.506092,0.513636,0.504167,0.518528,3
8,ravdess,Handcrafted MLP,4,0.537334,0.543939,0.534722,0.537916,3
9,ravdess,Handcrafted MLP,5,0.510962,0.505682,0.520833,0.517878,3


,dataset,model,macro_f1_mean,macro_f1_fold_std,accuracy_mean,accuracy_fold_std,uar_mean,uar_fold_std,weighted_f1_mean,weighted_f1_fold_std,n_outer_folds,mean_within_fold_seed_sd_macro_f1
0,ravdess,Concat Fusion MLP,0.943400,0.044329,0.946288,0.042708,0.947569,0.039168,0.946446,0.042752,5,0.006276
1,ravdess,Handcrafted MLP,0.510609,0.042700,0.515076,0.048927,0.513889,0.043899,0.515544,0.046122,5,0.016267
2,ravdess,Handcrafted SVM-RBF,0.516552,0.029226,0.528182,0.043882,0.512708,0.029356,0.528548,0.037224,5,NaN
3,ravdess,emotion2vec MLP,0.941686,0.037376,0.944545,0.035704,0.947083,0.031190,0.944940,0.035213,5,0.005125
4,resd,Concat Fusion MLP,0.591566,0.070635,0.623330,0.080916,0.619346,0.070711,0.627356,0.083935,5,0.028049
5,resd,Handcrafted MLP,0.167709,0.029044,0.191406,0.052091,0.197728,0.034057,0.179165,0.045921,5,0.013429
6,resd,Handcrafted SVM-RBF,0.172170,0.041332,0.190209,0.049594,0.200299,0.032464,0.183046,0.054327,5,NaN
7,resd,emotion2vec MLP,0.619723,0.083352,0.651828,0.093546,0.642024,0.087006,0.655154,0.096780,5,0.013265



Report macro_f1_mean ± macro_f1_fold_std as the speaker-partition sensitivity result.
Do NOT describe the SD across all fold×seed runs as an SD 'across folds'.
This notebook evaluates four utterance-level models; Cross-Attention is outside this supplementary CV.
